# OmegaPDF

Download manhwa panels from [OmegaScans](https://omegascans.org) and generate PDFs using the [OmegaAPI](https://omegaapi.vercel.app).

**Features:**
- Paste a URL to instantly download a chapter as PDF
- Concurrent panel downloads for speed
- Auto-retry on failed downloads
- Quality presets (Low / Medium / High DPI)
- Custom page ranges (e.g. pages 5-15 only)
- Merge multiple chapters into one PDF
- Batch download as separate PDFs
- Save directly to Google Drive
- Progress bars with tqdm
- Thumbnail preview before downloading
- Browse trending series
- Search for any series

**Quick Start:** Run setup, then use the "Download from URL" cell.

---

In [ ]:
#@title Setup — Install dependencies & imports { display-mode: "form" }
!pip install -q requests Pillow tqdm

import requests
import io
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, HTML

BASE_URL = "https://omegaapi.vercel.app"
API = f"{BASE_URL}/api/v1"
MAX_WORKERS = 8
MAX_RETRIES = 3
RETRY_BACKOFF = 1.0

QUALITY_PRESETS = {
    "low": {"dpi": 72, "label": "Low (72 DPI)", "desc": "Fast, small files"},
    "medium": {"dpi": 150, "label": "Medium (150 DPI)", "desc": "Balanced"},
    "high": {"dpi": 300, "label": "High (300 DPI)", "desc": "Print quality, large files"},
}

def api_get(path, params=None):
    r = requests.get(f"{API}{path}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def download_image_retry(url):
    """Download a single image with exponential backoff retry."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            return r.content
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout):
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(RETRY_BACKOFF * (2 ** attempt))

def download_images_concurrent(urls):
    """Download multiple images in parallel with a progress bar."""
    results = {}
    pbar = tqdm(total=len(urls), desc="Downloading panels", unit="panel")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(download_image_retry, url): i for i, url in enumerate(urls)}
        for future in as_completed(futures):
            idx = futures[future]
            results[idx] = future.result()
            pbar.update(1)
    pbar.close()
    return [results[i] for i in range(len(urls))]

def images_to_pdf(image_bytes_list, output_path, title=None, author=None, subject=None, dpi=150):
    """Convert image bytes to a single PDF with metadata."""
    pages = []
    for b in image_bytes_list:
        img = Image.open(io.BytesIO(b))
        if img.mode == "RGBA":
            img = img.convert("RGB")
        pages.append(img)

    pdf_info = {}
    if title: pdf_info["Title"] = title
    if author: pdf_info["Author"] = author
    if subject: pdf_info["Subject"] = subject

    first = pages[0]
    rest = pages[1:] if len(pages) > 1 else []
    first.save(
        output_path, format="PDF", save_all=True, append_images=rest,
        resolution=dpi, quality=90, pdf_info=pdf_info or None,
    )
    for img in pages:
        img.close()
    return output_path

def fetch_chapter(slug, chapter_slug):
    """Fetch chapter data from API. Returns (series_title, chapter_name, image_urls)."""
    data = api_get(f"/chapter/{slug}/{chapter_slug}")
    if not data.get("success"):
        raise RuntimeError(data.get("error", "Chapter not found"))
    info = data["data"]
    urls = info.get("images", [])
    if not urls:
        raise RuntimeError("No images found")
    series_title = info.get("series", {}).get("title", slug)
    ch_name = info.get("name", chapter_slug)
    return series_title, ch_name, urls

def safe_filename(text):
    return re.sub(r'[^\w\s-]', '', text).strip().replace(' ', '_')

print("Ready! All imports loaded.")

In [ ]:
#@title Download from URL — just paste and run! { display-mode: "form" }
url = "https://omegascans.org/series/manitto/chapter-82"  #@param {type:"string"}
quality = "medium"  #@param ["low", "medium", "high"]
page_range = ""  #@param {type:"string"}
#@markdown _Page range format: `5-15` (pages 5 through 15). Leave blank for all._
save_to_drive = False  #@param {type:"boolean"}
#@markdown _Check to save PDF to Google Drive (requires mounting)._

match = re.search(r"omegascans\.org/series/([^/]+)/chapter-(\d+)", url)
if not match:
    print("Invalid URL. Expected: https://omegascans.org/series/{slug}/chapter-{number}")
else:
    slug = match.group(1)
    chapter_slug = f"chapter-{match.group(2)}"
    series_title, ch_name, image_urls = fetch_chapter(slug, chapter_slug)

    # Apply page range
    if page_range.strip():
        parts = page_range.split("-")
        start, end = int(parts[0]), int(parts[1])
        image_urls = image_urls[max(0, start-1):end]
        print(f"Using pages {start}-{end}")

    print(f"Series: {series_title}")
    print(f"Chapter: {ch_name}")
    print(f"Pages: {len(image_urls)}")

    # Show thumbnail preview
    print("\nPreview (first page):")
    thumb = requests.get(image_urls[0], timeout=30).content
    thumb_img = Image.open(io.BytesIO(thumb))
    display(thumb_img.copy())
    thumb_img.close()

    # Download all panels
    dpi = QUALITY_PRESETS[quality]["dpi"]
    image_bytes = download_images_concurrent(image_urls)

    # Build PDF
    fname = f"{safe_filename(series_title)}_{chapter_slug}.pdf"
    images_to_pdf(
        image_bytes, fname,
        title=f"{series_title} — {ch_name}",
        author="OmegaPDF",
        subject=series_title,
        dpi=dpi,
    )
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"\nPDF ready: {fname} ({size_mb:.2f} MB, {len(image_urls)} pages)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = f"/content/drive/MyDrive/{fname}"
        import shutil
        shutil.copy2(fname, drive_path)
        print(f"Saved to Drive: {drive_path}")
    else:
        from google.colab import files
        files.download(fname)

In [ ]:
#@title Search for a series { display-mode: "form" }
query = "solo leveling"  #@param {type:"string"}

results = api_get("/search", params={"q": query})
if results.get("success") and results["data"]:
    for i, s in enumerate(results["data"][:10]):
        chapters = s.get("chaptersCount", "?")
        status = s.get("status", "")
        print(f"{i+1}. {s['title']}  [{status}] — {chapters} chapters  —  slug: {s['slug']}")
else:
    print("No results found. Try a different search term.")

In [ ]:
#@title Browse trending & popular series { display-mode: "form" }
page = 1  #@param {type:"integer"}
per_page = 15  #@param {type:"slider", min:5, max:50, step:5}

data = api_get("/series", params={"page": page, "perPage": per_page})
if data.get("success") and data["data"]:
    print(f"Page {page} — {len(data['data'])} series\n")
    for i, s in enumerate(data["data"], 1):
        views = s.get("totalViews", 0)
        views_str = f"{views/1_000_000:.1f}M" if views >= 1_000_000 else f"{views:,}"
        badge = f" [{s['badge']}]" if s.get('badge') else ""
        rating = s.get('rating', 0)
        chapters = s.get('chaptersCount', '?')
        status = s.get('status', '')
        print(f"{i:2d}. {s['title']}{badge}")
        print(f"    Rating: {rating} | Views: {views_str} | {chapters} ch | {status}")
        print(f"    slug: {s['slug']}")
        print()
else:
    print("No series found.")

In [ ]:
#@title List chapters for a series { display-mode: "form" }
series_slug = "solo-leveling"  #@param {type:"string"}

series = api_get(f"/series/{series_slug}")
if series.get("success"):
    data = series["data"]
    print(f"Title: {data['title']}")
    print(f"Status: {data.get('status', 'N/A')}")
    print(f"Chapters: {data.get('chaptersCount', len(data.get('chapters', [])))}")
    print(f"\nAvailable chapters:")
    chapters = data.get("chapters", [])
    for ch in chapters[:30]:
        free = "[Free]" if ch.get("isFree") else "[Paid]"
        print(f"  {ch['name']} {free}  —  slug: {ch['slug']}")
    if len(chapters) > 30:
        print(f"  ... and {len(chapters) - 30} more chapters")
else:
    print(f"Error: {series.get('error', 'Series not found')}")

In [ ]:
#@title Download chapter by slug { display-mode: "form" }
slug = "solo-leveling"  #@param {type:"string"}
chapter = "chapter-1"  #@param {type:"string"}
output_name = "Solo_Leveling_Ch1"  #@param {type:"string"}
quality = "medium"  #@param ["low", "medium", "high"]
page_range = ""  #@param {type:"string"}
save_to_drive = False  #@param {type:"boolean"}

series_title, ch_name, image_urls = fetch_chapter(slug, chapter)

if page_range.strip():
    parts = page_range.split("-")
    start, end = int(parts[0]), int(parts[1])
    image_urls = image_urls[max(0, start-1):end]
    print(f"Using pages {start}-{end}")

print(f"Series: {series_title}")
print(f"Chapter: {ch_name}")
print(f"Pages: {len(image_urls)}")

# Thumbnail preview
print("\nPreview (first page):")
thumb = requests.get(image_urls[0], timeout=30).content
thumb_img = Image.open(io.BytesIO(thumb))
display(thumb_img.copy())
thumb_img.close()

dpi = QUALITY_PRESETS[quality]["dpi"]
image_bytes = download_images_concurrent(image_urls)

fname = f"{output_name}.pdf"
images_to_pdf(
    image_bytes, fname,
    title=f"{series_title} — {ch_name}",
    author="OmegaPDF", subject=series_title, dpi=dpi,
)
size_mb = os.path.getsize(fname) / (1024 * 1024)
print(f"\nPDF ready: {fname} ({size_mb:.2f} MB, {len(image_urls)} pages)")

if save_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = f"/content/drive/MyDrive/{fname}"
    import shutil
    shutil.copy2(fname, drive_path)
    print(f"Saved to Drive: {drive_path}")
else:
    from google.colab import files
    files.download(fname)

In [ ]:
#@title Merge multiple chapters into one PDF { display-mode: "form" }
merge_slug = "solo-leveling"  #@param {type:"string"}
start_ch = 1  #@param {type:"integer"}
end_ch = 3  #@param {type:"integer"}
merge_quality = "medium"  #@param ["low", "medium", "high"]
save_to_drive = False  #@param {type:"boolean"}

dpi = QUALITY_PRESETS[merge_quality]["dpi"]
all_bytes = []
series_title = merge_slug

for ch_num in range(start_ch, end_ch + 1):
    ch_id = f"chapter-{ch_num}"
    print(f"Fetching {ch_id}...")
    try:
        series_title, ch_name, urls = fetch_chapter(merge_slug, ch_id)
        ch_bytes = download_images_concurrent(urls)
        all_bytes.extend(ch_bytes)
        print(f"  {ch_name}: {len(urls)} pages")
    except Exception as e:
        print(f"  Skipping {ch_id}: {e}")

if all_bytes:
    safe = safe_filename(series_title)
    fname = f"{safe}_ch{start_ch}-{end_ch}_merged.pdf"
    images_to_pdf(
        all_bytes, fname,
        title=f"{series_title} — Chapters {start_ch}-{end_ch}",
        author="OmegaPDF", subject=series_title, dpi=dpi,
    )
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"\nMerged PDF: {fname} ({size_mb:.2f} MB, {len(all_bytes)} pages)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = f"/content/drive/MyDrive/{fname}"
        import shutil
        shutil.copy2(fname, drive_path)
        print(f"Saved to Drive: {drive_path}")
    else:
        from google.colab import files
        files.download(fname)
else:
    print("No chapters downloaded.")

In [ ]:
#@title Batch download — separate PDFs { display-mode: "form" }
batch_slug = "solo-leveling"  #@param {type:"string"}
start_ch = 1  #@param {type:"integer"}
end_ch = 5  #@param {type:"integer"}
batch_quality = "medium"  #@param ["low", "medium", "high"]
zip_download = True  #@param {type:"boolean"}
#@markdown _Zip all PDFs into a single download._

dpi = QUALITY_PRESETS[batch_quality]["dpi"]
pdf_files = []

for ch_num in range(start_ch, end_ch + 1):
    ch_id = f"chapter-{ch_num}"
    print(f"\n{'='*40}")
    print(f"Processing {ch_id}...")
    try:
        series_title, ch_name, urls = fetch_chapter(batch_slug, ch_id)
        image_bytes = download_images_concurrent(urls)
        fname = f"{batch_slug}_{ch_id}.pdf"
        images_to_pdf(
            image_bytes, fname,
            title=f"{series_title} — {ch_name}",
            author="OmegaPDF", subject=series_title, dpi=dpi,
        )
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"  Saved: {fname} ({size_mb:.2f} MB, {len(urls)} pages)")
        pdf_files.append(fname)
    except Exception as e:
        print(f"  Error: {e}")

print(f"\nDone! {len(pdf_files)} PDFs created.")

if pdf_files:
    if zip_download:
        import zipfile
        zip_name = f"{batch_slug}_ch{start_ch}-{end_ch}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in pdf_files:
                zf.write(f)
        zip_mb = os.path.getsize(zip_name) / (1024 * 1024)
        print(f"Zip: {zip_name} ({zip_mb:.2f} MB)")
        from google.colab import files
        files.download(zip_name)
    else:
        from google.colab import files
        for f in pdf_files:
            files.download(f)